In [43]:
import requests
import os
from dotenv import load_dotenv
import datetime
import json

In [2]:
load_dotenv()

headers = {
    "User-Agent": os.getenv("USER_AGENT")
}

In [3]:
url = "https://api.chess.com/pub/leaderboards"

resp = requests.get(url, headers=headers)

In [4]:
resp.status_code

200

In [5]:
data = resp.json()


In [6]:
bullet_players = list()
rapid_players = list()

for i in range(50):
    bullet_players.append(data['live_bullet'][i]['username'])
    rapid_players.append(data['live_rapid'][i]['username'])


In [46]:
def get_response(player, ano, mes):
    url = f"https://api.chess.com/pub/player/{player}/games/{ano}/{mes:02d}"
    response = requests.get(url, headers=headers)
    return response

def save_data(data):
    now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S.%f")

    with open(f"../data/raw/{now}.json", 'w') as open_file:
        json.dump(data, open_file, indent=4)

In [22]:
resp = get_response("gmpiter", 2026, 4)
resp.status_code

200

In [23]:
data = resp.json()

In [47]:
save_data(data)

In [ ]:
url_matches = f"https://api.chess.com/pub/player/{player}/games/2026/04"

for player in bullet_players:
    resp = requests.get(url, headers=headers)
    if resp.status_code == 200:
        data = resp.json()

Agora com as funções prontas, e os melhores players de bullet e de rapid, podemos começar a coletar as partidas deles
1. Tenho três opções principais para fazer isso:
    - Example: https://api.chess.com/pub/player/erik/games/archives

        - archives: explorar todas as partidas deles
        - dessa forma iria vir todos os tipos de partidas, inclusive de outras modalidades
    - Example: https://api.chess.com/pub/player/erik/games/2009/10

        - ano-mes: coletar as partidas por meses
        - também continuaria vindo todas as patidas
    - Example: https://api.chess.com/pub/player/erik/games/live/180/2

        - por modalidade
        - me parece a mais útil agora, mas o problema é que o bullet tem várias submodalidades

Explorando os endpoints, vejo que nas partidas é possível filtrar por "time_class", sendo: daily, rapid, blitz ou bullet

então por enquanto vou usar a url ano-mes e depois posso alterar para archives, caso necessário

In [ ]:
def get_response(player, ano, mes):
    url = f"https://api.chess.com/pub/player/{player}/games/{ano}/{mes:02d}"
    response = requests.get(url, headers=headers)
    return response

def save_data(data):
    now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S.%f")

    with open(f"../data/raw/{now}.json", 'w') as open_file:
        json.dump(data, open_file, indent=4)

In [ ]:
resp = get_response('hikaru', 2026, 4)
resp.status_code

200

In [71]:
data = resp.json().get("games")
type(data[1])
# data[1]["time_class"]

dict

In [ ]:
def get_matches(player, ano, mes, time_class="bullet"):
    
    resp = get_response(player, ano, mes)

    if resp.status_code == 200:
        games = resp.json().get("games")
        split = [game for game in games if game.get("time_class") == time_class]
        return split
    else:
        print(f"ERRO {resp.status_code}: {resp.text}")
        return None


In [82]:
hikaru_blitz = get_matches("hikaru", 2026, 4)

Total de partidas: 177
Exemplo de partida: {'url': 'https://www.chess.com/game/live/166778575860', 'pgn': '[Event "Live Chess"]\n[Site "Chess.com"]\n[Date "2026.04.02"]\n[Round "-"]\n[White "Init0x7A"]\n[Black "Hikaru"]\n[Result "0-1"]\n[CurrentPosition "8/5p1k/P1R3pp/1p2P3/6P1/7P/2p5/1r4K1 w - - 1 47"]\n[Timezone "UTC"]\n[ECO "B06"]\n[ECOUrl "https://www.chess.com/openings/Modern-Defense-with-1-e4...3.Nf3-c6-4.c3-d6"]\n[UTCDate "2026.04.02"]\n[UTCTime "19:42:33"]\n[WhiteElo "2941"]\n[BlackElo "3411"]\n[TimeControl "180"]\n[Termination "Hikaru won by resignation"]\n[StartTime "19:42:33"]\n[EndDate "2026.04.02"]\n[EndTime "19:46:47"]\n[Link "https://www.chess.com/game/live/166778575860"]\n\n1. Nf3 {[%clk 0:03:00]} 1... g6 {[%clk 0:03:00]} 2. e4 {[%clk 0:02:58.6]} 2... Bg7 {[%clk 0:02:59.9]} 3. d4 {[%clk 0:02:58.1]} 3... c6 {[%clk 0:02:59.1]} 4. c3 {[%clk 0:02:56.1]} 4... d6 {[%clk 0:02:58]} 5. Bd3 {[%clk 0:02:55.1]} 5... Nf6 {[%clk 0:02:56.3]} 6. h3 {[%clk 0:02:54]} 6... O-O {[%clk 0:02

In [84]:
len(hikaru_blitz)

177